# Day 10 — Model Comparison & Selection

**Goal:** Benchmark Logistic Regression (Day 8 baseline) against Random Forest and XGBoost. Justify the final model choice based on performance *and* business trade-offs — not just the highest metric.

## Sections
1. Setup
2. Data Load + Feature Engineering
3. Train / Test Split
4. Train All Models (with timing)
5. Side-by-Side Metric Comparison
6. ROC Curve Overlay
7. Business Trade-off Analysis
8. Feature Importance (Best Model)
9. Engineered Features — Did They Help?
10. Final Model Selection

In [ ]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score, accuracy_score
)
from xgboost import XGBClassifier

from src.features.engineer import engineer_features

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42

## 2. Data Load + Feature Engineering

In [ ]:
for fname in ['credit_default_eda_ready.parquet',
              'credit_default_validated.parquet',
              'credit_default_cleaned.parquet']:
    path = PROJECT_ROOT / 'data/processed' / fname
    if path.exists():
        df_raw = pd.read_parquet(path)
        print(f'Loaded: {fname}  |  Shape: {df_raw.shape}')
        break

df_raw = df_raw.drop(columns=[c for c in ['edu_label','mar_label','age_group'] if c in df_raw.columns])

df = engineer_features(df_raw)
print(f'After engineering: {df.shape}  (+{df.shape[1] - df_raw.shape[1]} features)')

nan_counts = df.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]
if nan_cols.empty:
    print('NaN check: PASSED — no missing values in engineered features.')
else:
    print(f'NaN check: FAILED — {nan_cols.to_dict()}')

## 3. Train / Test Split

Same `random_state=42` and `stratify=y` as Day 8 — identical split so comparisons are fair.

In [ ]:
# Drop string column — keep numeric risk_segment_id
X = df.drop(columns=['default', 'risk_segment'])
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Class ratio for XGBoost scale_pos_weight
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.3f}')
print(f'scale_pos_weight for XGBoost: {scale_pos_weight:.2f}')

## 4. Train All Models (with timing)

In [ ]:
def train_and_time(model, X_tr, y_tr, X_te):
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    probs = model.predict_proba(X_te)[:, 1]
    preds = model.predict(X_te)
    infer_time = time.perf_counter() - t0

    return model, probs, preds, round(train_time, 3), round(infer_time * 1000, 1)  # ms for inference


# --- Logistic Regression (baseline, with scaling pipeline) ---
lr = Pipeline([('scaler', StandardScaler()),
               ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))])
lr, lr_probs, lr_preds, lr_train_t, lr_infer_t = train_and_time(lr, X_train, y_train, X_test)
print(f'LR    trained in {lr_train_t}s  |  inference {lr_infer_t}ms')

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                             n_jobs=-1, random_state=RANDOM_STATE)
rf, rf_probs, rf_preds, rf_train_t, rf_infer_t = train_and_time(rf, X_train, y_train, X_test)
print(f'RF    trained in {rf_train_t}s  |  inference {rf_infer_t}ms')

# --- XGBoost ---
# use_label_encoder removed in XGBoost 2.0 — do not pass it
xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
)
xgb, xgb_probs, xgb_preds, xgb_train_t, xgb_infer_t = train_and_time(xgb, X_train, y_train, X_test)
print(f'XGB   trained in {xgb_train_t}s  |  inference {xgb_infer_t}ms')

## 5. Side-by-Side Metric Comparison

In [ ]:
def metrics_row(name, y_true, y_pred, y_prob, train_t, infer_t):
    auc  = roc_auc_score(y_true, y_prob)
    gini = 2 * auc - 1
    return {
        'Model':      name,
        'Accuracy':   round(accuracy_score(y_true, y_pred), 4),
        'Precision':  round(precision_score(y_true, y_pred), 4),
        'Recall':     round(recall_score(y_true, y_pred), 4),
        'F1':         round(f1_score(y_true, y_pred), 4),
        'AUC-ROC':    round(auc, 4),
        'Gini':       round(gini, 4),
        'Train (s)':  train_t,
        'Infer (ms)': infer_t,
    }

comparison = pd.DataFrame([
    metrics_row('Logistic Regression', y_test, lr_preds,  lr_probs,  lr_train_t,  lr_infer_t),
    metrics_row('Random Forest',       y_test, rf_preds,  rf_probs,  rf_train_t,  rf_infer_t),
    metrics_row('XGBoost',             y_test, xgb_preds, xgb_probs, xgb_train_t, xgb_infer_t),
]).set_index('Model')

print(comparison.to_string())
print('\n** Recall is the priority metric — minimizes hidden losses (False Negatives)')

# Visual comparison
metrics_to_plot = ['Precision', 'Recall', 'F1', 'AUC-ROC', 'Gini']
plot_df = comparison[metrics_to_plot].T

fig, ax = plt.subplots(figsize=(12, 5))
plot_df.plot(kind='bar', ax=ax, width=0.7)
ax.set_ylim(0, 1)
ax.set_title('Model Comparison — Key Metrics', fontsize=14, fontweight='bold')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Model')
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=0.8, label='Random baseline')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/16_model_comparison.png', dpi=150)
plt.show()

## 6. ROC Curve Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name, probs, color in [
    ('Logistic Regression', lr_probs,  '#9C27B0'),
    ('Random Forest',       rf_probs,  '#FF9800'),
    ('XGBoost',             xgb_probs, '#2196F3'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1], [0,1], 'k--', lw=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/17_roc_curves_all_models.png', dpi=150)
plt.show()

## 7. Business Trade-off Analysis

> **Interview talking point:** "While XGBoost likely gives the best AUC, choosing a model requires weighing performance against interpretability, training cost, and regulatory requirements. In a Citi production environment, SR 11-7 requires that every model decision be explainable to a non-technical reviewer."

| Dimension | Logistic Regression | Random Forest | XGBoost |
|-----------|--------------------|--------------|---------|
| **Interpretability** | High — coefficients are direct | Medium — aggregate importance only | Low — black box by default |
| **SR 11-7 compliance** | Strong — native explainability | Moderate | Requires SHAP (Day 11) |
| **Performance** | Baseline | Better on non-linear patterns | Best on tabular data |
| **Training time** | Fastest | Moderate | Moderate |
| **Overfitting risk** | Low | Medium (depends on depth) | Moderate (tuning needed) |
| **Production recommendation** | Challenger / governance model | Not recommended for credit | **Primary** — with SHAP explanations |

## 8. Feature Importance (XGBoost)

In [ ]:
importance = pd.Series(
    xgb.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Top 25 — XGBoost
importance.head(25).plot(kind='barh', ax=axes[0], color='#2196F3')
axes[0].set_title('XGBoost — Top 25 Feature Importances', fontsize=12)
axes[0].set_xlabel('Importance (F-score)')
axes[0].invert_yaxis()

# Top 25 — Random Forest
rf_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)
rf_importance.head(25).plot(kind='barh', ax=axes[1], color='#FF9800')
axes[1].set_title('Random Forest — Top 25 Feature Importances', fontsize=12)
axes[1].set_xlabel('Importance (Mean Decrease in Impurity)')
axes[1].invert_yaxis()

plt.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/18_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 XGBoost features:')
print(importance.head(10).to_string())
print('\nEngineered features in top 10:')
engineered = [f for f in importance.head(10).index if f not in df_raw.columns]
print(engineered if engineered else 'None in top 10')

## 9. Engineered Features — Did They Help?

Retrain LR on raw features only vs. engineered features to quantify the value of Day 9 work.

In [ ]:
# Raw features only (same split, no engineering)
raw_cols = [c for c in df_raw.columns if c != 'default']
X_raw_train = X_train[raw_cols]
X_raw_test  = X_test[raw_cols]

lr_raw = Pipeline([('scaler', StandardScaler()),
                   ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))])
lr_raw.fit(X_raw_train, y_train)
lr_raw_probs = lr_raw.predict_proba(X_raw_test)[:, 1]

auc_raw = roc_auc_score(y_test, lr_raw_probs)
auc_eng = roc_auc_score(y_test, lr_probs)

print(f'LR on raw features only:        AUC = {auc_raw:.4f}')
print(f'LR on engineered features:      AUC = {auc_eng:.4f}')
print(f'Feature engineering delta:      +{auc_eng - auc_raw:.4f}')

xgb_raw = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                         scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                         random_state=RANDOM_STATE, n_jobs=-1)
xgb_raw.fit(X_raw_train, y_train)
xgb_raw_probs = xgb_raw.predict_proba(X_raw_test)[:, 1]

auc_xgb_raw = roc_auc_score(y_test, xgb_raw_probs)
auc_xgb_eng = roc_auc_score(y_test, xgb_probs)

print(f'\nXGBoost on raw features:        AUC = {auc_xgb_raw:.4f}')
print(f'XGBoost on engineered features: AUC = {auc_xgb_eng:.4f}')
print(f'Feature engineering delta:      +{auc_xgb_eng - auc_xgb_raw:.4f}')

## 10. Final Model Selection

In [ ]:
best_auc = comparison['AUC-ROC'].idxmax()
best_recall = comparison['Recall'].idxmax()

print('=== MODEL SELECTION SUMMARY ===')
print(comparison[['Recall', 'AUC-ROC', 'Gini', 'Train (s)', 'Infer (ms)']].to_string())
print(f'\nBest AUC-ROC:  {best_auc}')
print(f'Best Recall:   {best_recall}')
print()
print('DECISION:')
print('  Primary model   → XGBoost  (best AUC/Recall; SHAP explanations added Day 11)')
print('  Challenger model → Logistic Regression  (regulatory baseline; always maintained)')
print()
print('Next: Day 11 — SHAP explainability to make XGBoost SR 11-7 compliant.')